In [ ]:
!pip install -r requerients.txt

In [1]:
#Necesary libraries
import numpy as np
import cmath
import networkx as nx
import matplotlib.pyplot as plt
import itertools 
import json

from qiskit import QuantumRegister, ClassicalRegister, QuantumCircuit, transpile
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
from qiskit import transpile
from qiskit.visualization import  plot_histogram
from qiskit.quantum_info import state_fidelity,Statevector,partial_trace
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke,FakeBrisbane,FakeKyoto,FakeOsaka,FakeManilaV2,FakeKyiv
from qiskit.quantum_info import negativity




In [2]:
#Necesary functions:


#Create a matrix from Kronecker products of n-1 Identity matrices and 1 Hadamard matrix
def Cbase(n: int,p : int ,im=False ,norm=False ):
    """
    Inputs
    n: number of qubits of the system
    p: position of Haddamard matrix
    im: Boolean to decide if matrix S is applied
    norm: Boolean to decide if H colunms oh H are normalized (for combinatorial purpose only)
    Outputs:
    B: a (2^n)x(2^n) matrix produced by M_0(x).(x)M_j..(x)M_{n-1}, where M_j=Id_2 if j!=p and M_j= (S)^(int(im)) H if j=p
    """
    H=np.array([[1,1],[1,-1]]) #Define Haddamard matrix
    if norm: 
        H=1/np.sqrt(2)*H #Apply normalization for vectors
        
    Id=np.eye(2) # 2x2 identity matrix
    if im:
        S=np.array([[1,0],[0,1j]]) #If complex is implemented, then we define S matrix
        
    else:
        S=Id
    
    if p==1: #Tensor product for application of measurement basis Id is applied on all qubits q_j for j+1!=p 
        B=S@H #If j=p=1 we apply local tranformation S@H 
    else:
        B=Id 
    for j in range(1,n):
        if j+1==p: #Check for j+1=p to apply the desired local operation
            B=np.kron(B,S@H)
        else:
            B=np.kron(B,Id)
    return B

#Pairs of columns for measurements
def cols_pairs(q: int,p :int):
    """
    This function calculates the the pair of columns of a given basis of the set of the 2n bases such that they are used in
    the same polariation identity, i.e., the columns representing (|j>\pm \k>)/sqrt(2) from the basis.
    Inputs:
    q: number of qubits of the system
    p: positions of the Hadamard matrix
    
    Outputs: 
    pairs: a list of paired columns for the measurements operators
    """
    pairs=np.zeros((2**(q-1),2),dtype=int)
    n=q-p
    #print("n=",n)
    j=0
    if n==0:
        for col in range(0,2**q):
            if col % 2==0:
                pairs[j,0],pairs[j,1]=col,col+1
                j+=1
    else:
        Ind=np.ones((2**q))
        for col in range(0,2**q-2**n):
            #print("col=",col)
            if Ind[col]==1:
                #print("j=",j)
                pairs[j,0],pairs[j,1]=col,col+(2**n)
                #print(pairs[j])
                j+=1
                Ind[col],Ind[col+2**n]=0,0
                
    return pairs

#Ad Matrix

def AM(q: int):
    """
    Creates the adjacency matrix for an hyipercubic graph
    Input:
    q: number of qubits.
    Output
    A: adjacency matrix for Hypercube graph of qth grade
    
    """
    if q==1:
        A=np.array([[0,1],[1,0]],dtype=int)
    else:
        A=np.array(np.kron(np.eye(2),AM(q-1))+np.kron(AM(1),np.eye(2**(q-1))),dtype=int)
    
    return A

#Graph creation for hypercubic graph for n qubit system
def create_graph(n:int):
    """
    This function create a hypercubic graph for a n-qubit system
    Input:
    n: qubit number
    Output:
    G: Hypercube graph of degree n (Q_n)
    """
    edges=[]
    A=AM(n)
    
    for i in range(2**n):
        mat=A[i]
        for j in range(2**n):
            if mat[j]==1:
                temp=[i,j]
                edges.append(temp)
    vertices=[i for i in range(n)]
    G=nx.Graph()
    G.add_edges_from(edges)
    #nx.draw_networkx(G)
    #plt.show()
    return G

#All shorthest path between two nodes
def all_paths(n:int,s:int,t:int):
    """
    Find all paths with the lowest number of edges between nodes s and t of a graph
    Input:
    n: number of qubits
    s: source node
    t: target noude
    Output:
    paths: an array with all paths with the lowest number of edges between  nodes s and t
    """
    paths=[]
    for path in nx.all_shortest_paths(create_graph(n),source=s,target=t):
        paths.append(path)
    return paths

#Shorthest path wich maximes f=Sqrt(Sum(Abs(c_j)^2))
def maxpath(n:int,s:int,t:int, csq: np.array):
    """
    Calculates all paths with the lowest number of edges between nodes s and t of a graph. Then, selects the phat which 
    maximizes f= Sum_{path} |c_j|^2 for j\in Nodes(path)
    Input:
    n: number of qubits
    s: source node
    t: target noude
    csq: vector that contains |c_j|^2 with j=0,...,n-1
    Output:
    maxpath: the path with the maximun f
    """
    paths=all_paths(n,s,t)
    f=np.zeros(len(paths))
    for i,path in enumerate(paths):
        for j in path:
            f[i]+=csq[j]            
    maxpath=paths[np.argmax(f)]
    return maxpath

def all_max_paths(n:int,csq: np.array):
    """
    Calculate all paths that lead to node t from all others such that they maximize f
    Input:
    n: qubit number
    csq:  vector that contains |c_j|^2 with j=0,...,n-1
    Output:
    paths: all paths that lead to node t from all others such that they maximize f
    """
    paths=[]
    t=np.array(csq).argmax()
    for s in range(2**n):
        if s==t:
            pass
        else:
            paths.append(maxpath(n,s,t,csq))
    #print(paths)
    return paths

#Complete prob vectors with zeros
def full_prob_dist(n:int,pb: dict):
    """
    Fill zeros for an incomplete probability vector, i.e, for n-qubit system returns aa array of 2^n elements containing 
    all the probabilities for the 2^n elements of a basis, includig those with p_j=0
    Input: 
    n: number of qubits
    pb: a dictionary of a p dist from qiskit
    
    Output
    fpb= a fullfilled dictionay with porobability distribution with the null elements considered
    """
    fpb={}
    for i in list(itertools.product(['0','1'], repeat=n)):
        key="".join(i)
        if key in pb:
            fpb.update({key:pb[key]})
        else:
            fpb.update({key:0})
    return fpb

#Dataset post-processing to obtain results in the desired format
def post_process(qc_results : dict, n : int, p :int , im=False ):
    """
    Given that qiskit returns the probability distribution tagged as binary labels of the compuational basis, here we apply
    relabeling of the elements to represent the labels for a different basis from the 2n basis set.
    
    Input:
    qc_results: a dictionaray with the results of a quantum circuit experiment
    n: number of qubits
    p:position of H or SH matrix
    im: Bolean value to indicate if we use H o SH
    
    Output:
    MBs: a dictionay in the disered format for compute results
    """
    B=Cbase(n,p,im)
    kwds=[]
    for col in range(len(B[0])):
        for nz in np.nonzero(B[:,col]):
            #print(nz)
            kwd='p_'+str(nz[0])+'_'+str(nz[1])
        if int(np.real(B[nz[1],col]))==-1:
                kwd=kwd+'-'
        elif int(np.imag(B[nz[1],col]))==-1:
                kwd=kwd+'i-'
        elif int(np.imag(B[nz[1],col]))==1:
                kwd=kwd+'i+'
        else:
                kwd=kwd+'+'
        kwds.append(kwd)
        MBs=dict(zip(kwds,list(qc_results.values())))
    return MBs

#Determine de weighted adj matrix
def adj_w( n: int,my_dict):
    """
    Calculates all the polarization identitites and storage them as element of a matrix, a weighted adjacency matrix
    Input:
    n: qubit number
    my_dict= a dictonary with probability distribution for the 2n basis used for the tomography (Computational Basis excluded) 
    Output:
    Aw: Adj matrix with weights
    """
    Aw=np.zeros((2**n,2**n),dtype=complex)
    for i in range(len(Aw[0])):
        for j in range(i+1,len(Aw[0])):
            if 'p_'+str(i)+'_'+str(j)+'+' in my_dict:
                Aw[i,j]+=(my_dict['p_'+str(i)+'_'+str(j)+'+']-my_dict['p_'+str(i)+'_'+str(j)+'-']+
                          (my_dict['p_'+str(i)+'_'+str(j)+'i+']-my_dict['p_'+str(i)+'_'+str(j)+'i-'])*1j)
                Aw[j,i]=np.conjugate(Aw[i,j])
    #print(Aw)
    return Aw

def delta_adj(n : int,ns : int,my_dict):
    """
    Calculate error propagation for each polarization identity
    Input:
    n: qubit number
    my_dict= a dictonary with probability distribution for the 2n basis used for the tomography (Computational Basis excluded)
    ns: shot number (per basis)
    Output:
    Aw: Adj matrix with weights
    """
    delta_Aw=np.zeros((2**n,2**n),dtype=complex)
    for i in range(len(delta_Aw[0])):
        for j in range(i+1,len(delta_Aw[0])):
            if 'p_'+str(i)+'_'+str(j)+'+' in my_dict:
                # (p_ij^+ + p_ij^- - (p_ij^+ + p_ij^-)2+ 1j* ((q_ij^+ + q_ij^)-  (q_ij^+ - q_ij^-)^2))/ns
                p_real=((my_dict['p_'+str(i)+'_'+str(j)+'+'] + my_dict['p_'+str(i)+'_'+str(j)+'-'])-
                        (my_dict['p_'+str(i)+'_'+str(j)+'+'] - my_dict['p_'+str(i)+'_'+str(j)+'-'])**2)/ns
                
                p_imag=((my_dict['p_'+str(i)+'_'+str(j)+'i+'] + my_dict['p_'+str(i)+'_'+str(j)+'i-'])-
                        (my_dict['p_'+str(i)+'_'+str(j)+'i+'] - my_dict['p_'+str(i)+'_'+str(j)+'i-'])**2)/ns
                
                delta_Aw[i,j]+=p_real+1j*p_imag
                
                delta_Aw[j,i]=delta_Aw[i,j]
    return delta_Aw


def delta_pj(n: int, ns : int,cbp):
    """
    Calculate error propagation for probability distributions, considering multinomial distribution
    input:
    n: qubit number
    ns: shot number
    cbp: probbailities on computational basis
    Output
    delta_p: array with variance of pjs
    """
    delta_p=np.zeros(2**n)
    for j,pj in enumerate(cbp):
        delta_p[j]=(pj*(1-pj))/ns
    return delta_p

def delta_beta(n : int,Aw,dAw):
    
    """
    Calculate error propagation for phases of Λ_{jk}
    input:
    n: qubit number
    Aw: 2^n x 2^n Matrix containging all  polarization identity values
    dAw: 2^n x 2^n Matrix containging error for all polarization identity values
    Output
    delta_B: 2^n x 2^n Matrix array with variance of β_{jk}
    
    """
    delta_B=np.zeros((2**n,2**n))
    pairs=np.array(create_graph(n).edges)
    for pair in pairs:
        delta_B[pair[0],pair[1]]=((np.imag(Aw[pair[0],pair[1]])**2)*np.real(dAw[pair[0],pair[1]])+
                                  np.imag(dAw[pair[0],pair[1]])*(np.real(Aw[pair[0],pair[1]]))**2)/np.abs(Aw[pair[0],pair[1]])**4
        delta_B[pair[1],pair[0]]=delta_B[pair[0],pair[1]]
    return delta_B

def delta_alpha(n : int,paths,delta_B):
    """
    Estimate error propagation for phases of c_{j}e^{α_j}
    input:
    n: qubit number
    phaths: arry with all optimal path to a given node k
    delta_B: 2^n x 2^n Matrix array with variance of β_{jk}
    
    Output
    delta_phase: array with variance of all α_j 
    
    """
    delta_phase=np.zeros(2**n)
    for path in paths:
        pairs= [[path[i],path[i+1]] for i in range(len(path)-1)]
        #print(pairs)
        for pair in pairs:
            delta_phase[path[0]]+=delta_B[pair[0],pair[1]]
            
    return delta_phase

def fidelity_with_error(n:int, cbp, delta_probj, delta_alphas, estimated_state, teoretical_state):
    """
    Calculate fidelity between an estimated pure quantum state and the theoretical test state
    
    Input:
    n : qubit number
    cbp: Array with probability distribution on the computational basis
    delta_probj: array with variance of probabilities
    delta_alpha: array with variance of all α_j 
    estimated_state: state obtained from tomographic protocol
    teoretical_state: state ideally prepared
    
    Output:
    res: tuple with fidelity f +(-) Δf
    """
    delta_f=0
    gamma=[cmath.phase(teoretical_state[i]) for i in range(len(teoretical_state))]
    vec_a=[np.abs(teoretical_state[i]) for i in range(len(teoretical_state))]
    alphas=[cmath.phase(estimated_state[i]) for i in range(len(estimated_state))]
    
    for j in range(2**n):
        delta_fj=((vec_a[j]**2)**2)*delta_probj[j]
        for l in range(2**n):
            if l==j:
                continue
            else:
                delta_fj+=(vec_a[j]/np.sqrt(cbp[j]))*vec_a[l]*np.sqrt(cbp[l])*np.cos((gamma[j]-gamma[l])-(alphas[j]-alphas[l]))**2
                #print(delta_fj)
        delta_f+=((delta_fj)**2)*delta_probj[l]
        
        delta_fj=0
        
        for k in range(2**n):
            if k==j:
                continue
            else:
                if k<=j-1:
                    delta_fj-=vec_a[k]*np.sqrt(cbp[k])*np.sin((gamma[j]-gamma[l])-(alphas[j]-alphas[l]))
                else:
                    delta_fj+=vec_a[k]*np.sqrt(cbp[k])*np.sin((gamma[j]-gamma[l])-(alphas[j]-alphas[l]))
                #print(delta_fj)
        
                    
        delta_f+=2*vec_a[j]*np.sqrt(cbp[j])*delta_alphas[j]*(delta_fj)**2
        #print(delta_f)
    #print(delta_f)
    res=(state_fidelity(teoretical_state,estimated_state) , delta_f)
    #print(res)
    return res

#Determine al the c[i] complex coeficients of qs
def qto(n,Aw,ct,the_paths):
    """
    Input:
    n: qubit number
    Aw: Adj matrix with weights
    ct: an array with the prob dist in computacional basis
    the paths: an array with all optimal paths 
    Output:
    c: an array with probability amplitudes of the reconstructed state
    """
    c=np.ones(2**n,dtype=complex)
    
    impar=0
    par=1
    #print(all_max_paths(n,ct))
    for path in the_paths:
        #print(path)
        #print((len(path)-1) % 2==1)
        for j in range(len(path)-1):
            if j%2==0:
                
                c[path[0]]= c[path[0]]*np.conjugate(Aw[path[j],path[j+1]])
                
            else:
    
                c[path[0]]=c[path[0]]/ (Aw[path[j],path[j+1]])
        #print(c[path[0]])    
        if (len(path)-1) % 2==1:            
            impar+=(np.absolute(c[path[0]])**2)
            #print(impar)
        else:
            
            par+=(np.absolute(c[path[0]])**2)
            #print(par)
    #print(c)
    #print(impar)
    #print(par)
    ctilde=np.roots([par,-1,impar/4])
    #print(ctilde)
    c[np.argmax(ct)]=np.sqrt(np.max(ctilde.real))
    
    for path in the_paths:
        #print(path)
        if (len(path)-1) % 2==1:
            c[path[0]]/=(2*c[np.argmax(ct)])
        else:
            c[path[0]]*=c[np.argmax(ct)]
    return c


def state_3layers(nq: int):
    """
    Entangled state with 3 layers
    input:
        nq= qubit number
    Output:
        qc: quantum circuit with 3 layer state
    """
    qr=QuantumRegister(nq,'qr')
    cr=ClassicalRegister(nq,'cr')
    qc=QuantumCircuit(qr,cr)
    if nq%2==0:
        qc.sx(list(range(nq)))
    else:
        qc.sx(list(range(nq-1)))
        qc.x(nq-1)
    for i in range(0,nq-1,2):
        qc.ecr(i,i+1)
    for i in range(1,nq-1,2):    
        qc.ecr(i,i+1)
    #qcf5.sx(0)
    qc.sx(0)
    #for i in range(1,nq):
    #    qc.sx(i)
    return qc

def state_4layers(nq):
    """
    Entangled state with 4 layers (This was used for the test on Kyoto QPU)
    input:
        nq= qubit number
    Output:
        qc: quantum circuit with 4 layer state
    """
    qr=QuantumRegister(nq,'qr')
    cr=ClassicalRegister(nq,'cr')
    qc=QuantumCircuit(qr,cr)
    qc.sx(list(range(nq)))
    for i in range(0,nq-1,2):
        qc.ecr(i,i+1)
    for i in range(1,nq-1,2):    
        qc.ecr(i,i+1)
    #qcf5.sx(0)
    for i in range(1,nq):
        qc.sx(i)
    return qc


#Circuit Functions
def circuit_cons(nq,pH,primada,qci):
    ################### Args #####################################################################
    """
    This function creates all the circuits to implement the 2n+1 basis method given an initial state qci 
    Input:
    # nq: Qubit number.
    # pH Haddamard gate position.
    # primada: if 0 computational bases, if 1 only H gate is applied, valor 2 apply H Sdagger operations (inverse of SH) 
    # qci: initial state.
    
    Output:
    prop_dist: prob dist the desired state and measurement.
    """
    ########################## Circuit for bases ######################################################
    qr = QuantumRegister(nq, 'Qubit')
    cr = ClassicalRegister(nq, 'Bit')
    qc= QuantumCircuit(qr,cr)  
   
    ########################### Computational basis ################################################
    qr = QuantumRegister(nq, 'Qubit')
    cr = ClassicalRegister(nq, 'Bit')
    cb = QuantumCircuit(qr,cr)  
    
    ########################### Resultado Base computacional ################################################
    if primada == 0: 
        cb.measure(range(nq),range(nq))
        combined  = qci.compose(cb)       
        return combined
    ###########################################################################################################
    elif primada == 1:
        if pH <= nq:  
            cb.h(nq-pH) # as Qiskit counts from right to left we need to change position of H gate
            
            cb.measure(qr,cr)            
            combined = qci.compose(cb)
            return combined

    elif primada == 2:
        if pH <= nq:
            cb.sdg(nq-pH)
            cb.h(nq-pH)
            cb.measure(qr,cr)
            combined = qci.compose(cb)
            return combined
    else:
        print(f'ERROR look qubit number, H position, primada value or initial state')
        
def q_tomo_sim_results_w_err(n,ns, sim_result,teoretical_state):
    """
    Tomography using previous experimental results
    Input:
    - nq: number of qubits
    - ns: number of shots
    - sim_results: dictionary with results from ALL measurements for nq qubits
    - teoretical_state: state ideally prepared 
    """
    
    
    csq=full_prob_dist(n,sim_result[0])
    
    pds={}
    
    config=[]
    for pH in range(1,n+1):
        for primada in [1,2]:
            config.append([pH,primada])
            
    
    #Post proecessing of measurements
    for i in range(0,len(config)):
        #Complete the prob dist
        
        mb=full_prob_dist(n,sim_result[i+1])
        mb={k: v / ns for k, v in mb.items()}
        pH,primada=config[i]
        
        pds.update(post_process(mb,n,pH,primada==2))
    
    Aw= adj_w(n,pds)
    dAw=delta_adj(n,ns,pds)
    #optimal paths
    
    csqv=list(csq.values())
    pjs=[csqv[i]/ns for i in range(len(csqv))]
    
    the_paths=all_max_paths(n,csqv)
    
    state=qto(n,Aw,csqv,the_paths)
    state=state/np.linalg.norm(state)
    phases=[]
    for i in range(len(state)):
        phases.append(cmath.phase(state[i]))
    
    f_state=[]
    for i in range(len(state)):
        f_state.append(np.sqrt(csqv[i]/ns)*np.exp(1j*phases[i]))
    f_state=f_state/np.linalg.norm(f_state)
    
    
    #Error calc
    delta_B=delta_beta(n,Aw,dAw)
    
    delta_alphas=delta_alpha(n,the_paths,delta_B)
    delta_probj=delta_pj(n, ns,np.array(csqv)/ns)
    
    
    (fidel , delta_fidelity)=fidelity_with_error(n, pjs, delta_probj, delta_alphas, f_state, teoretical_state)  
    
    return f_state,csq,pds,phases, the_paths, fidel, delta_fidelity , Aw, dAw, delta_B,delta_alphas,delta_probj,csqv

# Execution examples

## Kyoto

In [4]:
ns=20000
my_res_kyoto={}
for nq in range(3,9):
    state=state_4layers(nq)
    with open(f'res_kyoto_{nq}.txt') as f:
        sim_res = json.load(f)
    #sim_res=test_job.result().get_counts()
    res_tmp=q_tomo_sim_results_w_err(nq,ns,sim_res,Statevector(state))
    #print(res_tmp[5],' ',res_tmp[6])
    #print(f'Kyoto {nq} qubits: ',state_fidelity(Statevector(state),Statevector(res_tmp[0])))
    my_res_kyoto.update({f'{nq}':(res_tmp[5],res_tmp[6])})
    print(f'Kyoto {nq} qubits: {res_tmp[5]} ± {res_tmp[6]}')

Sher 3 qubits: 0.9916208837726312 ± 3.030418945696019e-05
Sher 4 qubits: 0.9970662957598613 ± 4.6579489211661065e-05
Sher 5 qubits: 0.9899326476233214 ± 6.297950567132665e-05
Sher 6 qubits: 0.9868714214177463 ± 0.00012638932483078502
Sher 7 qubits: 0.9792535502081341 ± 0.0004116247355231918
Sher 8 qubits: 0.9137075866163903 ± 0.0024040058778192455


## Save to a file

In [ ]:
with open('res_kyo_w_err.txt', 'w') as filehandle:
    json.dump(my_res_kyoto, filehandle)

## Kyiv

In [8]:

ns=20000
my_res_kyiv={}
for nq in range(3,11):
    state=state_3layers(nq)
    with open(f'res_kyiv_{nq}.txt') as f:
        sim_res = json.load(f)
    #sim_res=test_job.result().get_counts()
    res_tmp=q_tomo_sim_results_w_err(nq,ns,sim_res,Statevector(state))
    #print(res_tmp[5],' ',res_tmp[6])
    #print(f'Kyoto {nq} qubits: ',state_fidelity(Statevector(state),Statevector(res_tmp[0])))
    my_res_kyiv.update({f'{nq}':(res_tmp[5],res_tmp[6])})
    print(f'Kyiv {nq} qubits: {res_tmp[5]} ± {res_tmp[6]}')

Kyiv 3 qubits: 0.986726311461195 ± 4.045440127719832e-05
Kyiv 4 qubits: 0.9970002882002368 ± 4.0453050559150434e-05
Kyiv 5 qubits: 0.9919685302791336 ± 5.468575250213856e-05
Kyiv 6 qubits: 0.9837681427930562 ± 8.252273554655863e-05
Kyiv 7 qubits: 0.9799983491567625 ± 0.00015005679601256766
Kyiv 8 qubits: 0.9246338455547211 ± 0.011243877893890517
Kyiv 9 qubits: 0.8306951642269923 ± 0.009455582904493515
Kyiv 10 qubits: 0.9179216007819881 ± 0.009546899785603961


In [ ]:
with open('res_kyiv_w_err.txt', 'w') as filehandle:
    json.dump(my_res_kyiv, filehandle)

## Sherbrooke

In [7]:
import json
ns=20000
my_res_sherbrooke={}
for nq in range(3,10):
    state=state_3layers(nq)
    with open(f'res_sherbrooke_{nq}.txt') as f:
        sim_res = json.load(f)
    #sim_res=test_job.result().get_counts()
    res_tmp=q_tomo_sim_results_w_err(nq,ns,sim_res,Statevector(state))
    #print(res_tmp[5],' ',res_tmp[6])
    #print(f'Kyoto {nq} qubits: ',state_fidelity(Statevector(state),Statevector(res_tmp[0])))
    my_res_sherbrooke.update({f'{nq}':(res_tmp[5],res_tmp[6])})
    print(f'Sher {nq} qubits: {res_tmp[5]} ± {res_tmp[6]}')

Sher 3 qubits: 0.9872727878519166 ± 3.6047430610410335e-05
Sher 4 qubits: 0.9836217803257759 ± 6.677623110519765e-05
Sher 5 qubits: 0.9889855203101148 ± 6.922202391595354e-05
Sher 6 qubits: 0.9854302478943172 ± 0.00012787589045692662
Sher 7 qubits: 0.9706740894603926 ± 0.00030533689296017766
Sher 8 qubits: 0.9462625514392958 ± 0.0009369265914169613
Sher 9 qubits: 0.9271965193370836 ± 0.003964938812558004


In [ ]:
with open('res_sherbrooke_w_err.txt', 'w') as filehandle:
    json.dump(my_res_sherbrooke, filehandle)